# Assignment 2: Datasets, DataLoaders & Convolutional Neural Networks

Welcome to your second assignment!

This assignment is split into two parts:

**Part 1 — Ice Cream Sales (Revisited)**  
We return to the ice cream dataset from Assignment 1, but this time the data is already
pre-processed and all 16 features are included. The focus shifts to **structuring the
training pipeline** properly:
- Implement a `Dataset` and `DataLoader` using PyTorch's built-in abstractions
- **Training, Validation, Test Subsets** — Learn why a good train / validation / test split matters
- **Monitor Training** — Use a live loss plot to monitor the training in real time

**Part 2 — Image Classification with CNNs**  
We move from tabular data to images and introduce Convolutional Neural Networks (CNNs).
Using the MNIST handwritten-digit dataset, you will:
- **Explore Modular Model Design** — Avoid writing redundant code by defining own classes for repeated blocks
- **Build an Image Classifier** — Classify images with a simple MLP using `nn.Flatten`
- **Build another Image Classifier** — Compare the first classifier with a CNN
- **Understand how CNNs work** — Visualise the feature maps learned by the convolutional filters

> **Datasets**  
> - **Part 1:**</br>`ice_cream_sales_augmented.csv` — 3 years of pre-processed daily sales data (16 features)  
> - **Part 2:**</br>
MNIST — 70 000 28 × 28 pixel greyscale images of handwritten digits (0–9)</br>
CIFAR10 - 60 000 32 x 32 pixel colour images in 10 classes

## Table of Contents

**Part 1 — Ice Cream Sales**
- [Imports](#imports)
- [1 — Load Data](#1)
- [2 — Dataset & DataLoader](#2)
    - [2.1 — Implement custom `IceCreamDataset`](#2-1)
    - [2.2 — Create Dataset](#2-2)
    - [2.3 — Create Subsets for Training, Validation and Testing](#2-3)
    - [2.4 — Create DataLoaders](#2-4)
- [3 — Model Design](#3)
- [4 — Training](#4)
    - [4.1 — Train on the Temporal Split](#4-1)
    - [4.2 — Discussion: Why did Validation Loss stay high?](#4-2)
    - [4.3 — Retrain with a Random Split](#4-3)

**Part 2 — Image Classification with CNNs**
- [5 — Introduction to Image Data](#5)
    - [5.1 — Define Training Loop Template](#5-1)
- [6 — Baseline: MLP with `nn.Flatten`](#6)
    - [6.1 — Train the MLP](#6-1)
- [7 — CNN: Architectural Bias](#7)
    - [7.1 — Implement the CNN](#7-1)
    - [7.2 — Train the CNN](#7-2)
    - [7.3 — Compare Parameter Counts](#7-3)
- [8 — Modular Model Design](#8)
    - [8.1 — Implement the Modular CNN](#8-1)
    - [8.2 — Train the Modular CNN](#8-2)
- [9 — Feature Map Visualisation](#9)
- [Bonus — Label Manipulation](#bonus)
- [Conclusion](#conclusion)


<a name="imports"></a>
## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import clear_output

# For Part 2
import torchvision
import torchvision.transforms as transforms

<a name="1"></a>
## 1 — Load Data

The CSV already contains one row per day with all 16 features and the normalised target.
The column `units_sold_norm` is the normalised version of `units_sold` — this is what
the model will predict. The original `units_sold`, `target_mean` and `target_std` columns
are kept so you can reverse the normalisation for interpretation.


In [ ]:
FILE_PATH = 'https://raw.githubusercontent.com/opencampus-sh/course-material/main/applied-machine-learning/week-02/augmented_ice_cream_sales.csv'

df = pd.read_csv(FILE_PATH, parse_dates=['date'])

# Set the number of rows you want to display.
rows_to_display = 10

# Display the shape and first rows of the DataFrame.
print(f"Shape: {df.shape}")
df.head(rows_to_display)

<a name="2"></a>

## 2 — Dataset & DataLoader

In the first assignment, you trained the model on the full dataset, but this time, you will split the full dataset into subsets for training, validation and testing.</br>
You also passed the feature and target tensor for the entire `1096` days directly into the training loop. This works for small datasets, but it does not scale well.</br>
When training more complex models on larger datasets, we therefore train the model on batches of data in an approach called **Mini-Batch Gradient Descent**.

> **Splitting the data subsets** for training, validation and testing and the **training on batches** are two different techniques, that usually go hand-in-hand.
> - You could split your dataset into subsets, but train the model on the whole subset at once (**Full Gradient Descent**)
> - You could also simply divide your full dataset into batches without splitting it into subsets for training, validation and testing

The difference between **Full Gradient Descent** and **Mini-Batch Gradient Descent** is when we update the parameters of the model:
- With **Full Gradient Descent**, we update the weights and biases only once at the end of every epoch after the model has seen all of the training data
- With **Mini-Batch Gradient Descent**, we update the weights and biases after every batch of training data and have multiple updates during a single epoch

While **Mini-Batch Gradient Descent** updates the parameters of the model after it has only seen a fraction of the training data, it is usually considered "gold-standard" for scalable and robust training.

In the last assignment, you used **Full Gradient Descent** to train the model, so here we will use **Mini-Batch Gradient Descent**.

PyTorch provides two classes to support the split into subsets for training, validation and testing and the batching of these subsets:

| Class        | Responsibility                                                              |
|--------------|-----------------------------------------------------------------------------|
| `Dataset`    | Knows *what* the data is and how to retrieve one sample by index            |
| `DataLoader` | Wraps a `Dataset` and handles *batching*, *shuffling*, and parallel loading |

</br>

When training a model, we can simply pass a `Dataset` (or `Subset`) to PyTorch's default `DataLoader` class and let it handle the *batching*, *shuffling*, and parallel loading for us. </br>
For the `Dataset` however, PyTorch can only give us a general high-level template and basic structure and we need to create our own version of the `Dataset` class to make sure it works for our data.

> In **Object-Oriented Programming** terms, we say we are _"extending PyTorch's default `Dataset` class"_.</br>
> It's a way of saying we copy everything from PyTorch's default `Dataset` class and just add a bit of extra logic to deal with the data in our specific case.

For the custom `Dataset`, we always add the extra logic by implementing the following three methods:
> - `__init__()`: Setup of the general information and configuration of the dataset (data path, transforms, etc.)
> - `__len__()`: Returns the total number of samples (or pairs) in the dataset
> - `__getitem__()`: Applies transforms and retrieves a single sample at the given index

<a name="2-1"></a>
### 2.1 — Implement custom `IceCreamDataset`</br><font color="white"> 2.1 — </font><small>(by extending PyTorch's default `Dataset` class)</small>

In [ ]:
class IceCreamDataset(Dataset):

    SEASON_MAP = {1: 'Winter', 2: 'Spring',  3: 'Spring', 4: 'Spring', 5: 'Summer', 6: 'Summer', 7: 'Summer', 8: 'Autumn', 9: 'Autumn', 10: 'Autumn', 11: 'Winter', 12: 'Winter'}

    def __init__(self, file_path, transform=None):
        
        df = pd.read_csv(file_path, parse_dates=['date'])

        self.data = torch.tensor(df.loc[:, df.columns != 'date'].values.astype(np.float32), dtype=torch.float32)
        self.dates = df['date']

        self.transform = transform

    ### START CODE HERE ###

    def __len__(self):
        return None                                # Return the total number of samples in the dataset

    def __getitem__(self, idx):
        sample = None                              # Retrieve the sample at index `idx` from `self.data`

        if self.transform:
            sample = self.transform(sample)

        features = None                            # Extract the features for the current sample (first 16 columns)
        target = None                              # Extract target value for the current sample (last column)
        
        target = target.unsqueeze(0)               # The predictions of the model will have shape (..., 1), so we need to reshape the target to match that shape as well.

        return None, None                          # Return the features and target as a tuple

    ### END CODE HERE ###

In [ ]:
class Normalize:
    def __init__(self, feature_mean, feature_std):
        self.feature_mean = torch.tensor(feature_mean, dtype=torch.float32)
        self.feature_std  = torch.tensor(feature_std,  dtype=torch.float32)

    def __call__(self, x):
        return (x - self.feature_mean) / (self.feature_std)

In [ ]:
# We don't want to normalize boolean columns and therefore use mean=0, std=1 resulting in a no-operation normalisation (values stay 0/1).
feature_and_target_mean = np.array([df[col].mean() if col_type != bool else 0 for col, col_type in zip(df.columns, df.dtypes) if col != 'date'])
feature_and_target_std = np.array([df[col].std() if col_type != bool else 1 for col, col_type in zip(df.columns, df.dtypes) if col != 'date'])
feature_and_target_std[feature_and_target_std == 0] = np.finfo(np.float32).eps  # Avoid division by zero for constant columns

<a name="2-2"></a>
### 2.2 Create Dataset
Create a dataset with `transform = Normalize(feature_and_target_mean, feature_and_target_std)` and access a few samples using `dataset[idx]`.

In [ ]:
### START CODE HERE ###

transform = None

### END CODE HERE ###

dataset = IceCreamDataset(FILE_PATH, transform)

print(f"Dataset length: {len(dataset)}")

<a name="2-3"></a>
### 2.3 — Create Subsets for Training, Validation and Testing

In [ ]:
from torch.utils.data import Subset

def temporal_split(dataset):
    return (Subset(dataset, np.nonzero(df.date.dt.month.isin([1, 2, 3, 4, 8, 9, 10, 11, 12]) & (df.date.dt.year <= 2023))[0]),
            Subset(dataset, np.nonzero(df.date.dt.month.isin([5, 6, 7]) & (df.date.dt.year <= 2023))[0]),
            Subset(dataset, np.nonzero(df.date.dt.year == 2024)[0]))

In [ ]:
train_set, val_set, test_set = temporal_split(dataset)

print(f"Train: {len(train_set):>4d} samples")
print(f"Val:   {len(val_set):>4d} samples")
print(f"Test:  {len(test_set):>4d} samples")
print(f"Total: {len(train_set)+len(val_set)+len(test_set):>4d} samples")

<a name="2-4"></a>
### 2.4 — Create DataLoaders

Wrap each subset in a `DataLoader` and set `batch_size=BATCH_SIZE` for the `train_loader`.

- `shuffle=True` for the training loader (randomises the order of batches each epoch)
- `shuffle=False` for validation and test (order does not matter for evaluation)


In [ ]:
BATCH_SIZE = 32

### START CODE HERE ###

train_loader = None
val_loader   = None
test_loader  = None

### END CODE HERE ###

# Inspect one batch
features_batch, targets_batch = next(iter(train_loader))
print(f"Feature batch shape: {features_batch.shape}")
print(f"Target  batch shape: {targets_batch.shape}")

---
> #### 🤔 **Discussion:** What does a `Dataloader` class actually do?
>
> Take a look at the shape of the features and the target value for a single batch.</br>
> During training, we will pass the features in this shape into to model, so it makes sense to understand the dimension and how the `DataLoader` class operates.


`### ADD YOUR DISCUSSION AND IDEAS HERE ###`

<details>
  <summary><b>Answer</b> (click to expand)</summary>

> The shape of each batch corresponds to `(BATCH_SIZE, N_features)`.
>
> The `DataLoader` class simply extracts individual pairs from your custom `IceCreamDataset` and stacks them together.
>
> A single pair consists of 
> - feature values with shape `torch.Size(N_features)` and 
> - target values with shape `torch.Size([1])`
> 
> Stacking the pairs together creates the batch dimension.
> 
> In the first assignment, the features and target values had a very similar shape (`(N_days, N_features)` and `(N_days, 1)`), because the whole dataset was a single batch (`BATCH_SIZE = N_days`)
</details>

<a name="3"></a>
## 3 — Model Design

In the lecture, we discussed, that we need to define <u>**three components**</u> before we can train a model, let's define the model first.

In the last assignment, the training initially failed because the target values were not
normalised. To fix this, you had to redefine the entire model from scratch — there was
no clean way to reset the weights without rewriting the code.

This time, we define a template **model class** so that calling `model = IceCreamModel()` always
gives you a fresh, randomly initialised model. This pattern is standard practice and will
carry over directly to the CNN section in Part 2.

In the first assignment, you defined the model via `model = nn.Sequential(...)`. Here, we will define the model as `model = IceCreamModel()`.</br>
When defining our own model class by extending the `nn.Module()` class, we need to implement the following methods:
> - `__init__()`: Defines the layers and architecture of the model
> - `forward()`: Defines how data flows through the model (forward pass)

Under the hood, also `nn.Sequential()` defines these two methods, we just never have to change or worry about them.

The architecture is the same as in Assignment 1 and has three `nn.Linear` layers, each followed by a `nn.ReLU()` activation function, except for the last one:
- **Input Layer**: An `nn.Linear` layer that accepts **`N_INPUT_FEATURES` input features** (= 16 in our case) and outputs **64 features**.
- **Hidden Layer**: An `nn.Linear` layer that takes the **64 features** from the previous layer and outputs **32 features**.
- **Output Layer**: A final `nn.Linear` layer that takes the **32 features** from the hidden layer and produces a **single output** value.

In [ ]:
N_INPUT_FEATURES = 16

# Define the template for the model architecture.
class IceCreamModel(nn.Module):

    def __init__(self):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(N_INPUT_FEATURES, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# Create an untrained instance of the model with fresh, randomly initialised weights and biases.
model = IceCreamModel()

Now that we have the template **model class** and also already an untrained instance of the model, let's define the <u>**remaining two building blocks**</u>:

In [ ]:
loss_fn   = nn.MSELoss()

In [ ]:
LR = 0.01
optimizer = optim.SGD(model.parameters(), lr=LR)

<a name="4"></a>
## 4 — Training

### 4.1 — Train on the Temporal Split

<a name="4-1"></a>

As for the first assignment, we train the model for 2 000 epochs.</br>
This time however, we not only train the model, but also validate the training using our validation set.

Instead of printing the loss every 50 epochs, we use a live plot, that gets updated every `PLOT_EVERY` epochs to keep track of the progress.

Note that we use **Mini-Batch Gradient Descent** here and update the parameters after every batch during training.

Each epoch therefore consists of the following phases: 
- **Train phase** — execute the <u>**five training steps**</u> per batch in the `train_set` by iterating over `train_loader`  
    1. Zero the gradients — `optimizer.zero_grad()`
    2. Forward pass — `outputs = model(features)`
    3. Compute the loss — `loss = loss_function(outputs, targets)`
    4. Backward pass — `loss.backward()`
    5. Update the parameters — `optimizer.step()`
- **Validation phase** — processes the batches in `val_set` by iterating over `val_loader` inside `torch.no_grad()`
- **Logging** — append the mean epoch losses to `train_losses` and `val_losses`
- **Live plot** — every `PLOT_EVERY` epochs, redraw the loss curves using `clear_output`

> 💡 Mean loss per epoch = sum of all batch losses / number of batches  
> Use `len(train_loader)` and `len(val_loader)` as the denominators.

In [ ]:
N_EPOCHS   = 2_000

# Redraw the loss plot every `PLOT_EVERY` epochs
PLOT_EVERY = 50

mean_train_loss_per_epoch = []
mean_val_loss_per_epoch = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()

    total_train_loss_during_epoch = 0.0

    ### MODEL TRAINING LOOP ###
    # Mini-Batch Gradient Descent: Iterate over the training data in batches and update the model parameters after each batch.
    for training_batch in train_loader:
        features, targets = training_batch

        ### START CODE HERE ###

        # 1. Zero gradients
        None

        # 2. Forward pass
        outputs = None

        # 3. Compute loss
        loss = None

        # 4. Backward pass
        None

        # 5. Update parameters
        None

        ### END CODE HERE ###

        total_train_loss_during_epoch += loss.item()

    # Compute the mean loss by dividing by the number of batches (given by `len(train_loader)`).
    mean_train_loss_per_epoch.append(total_train_loss_during_epoch / len(train_loader))

    ### MODEL VALIDATION LOOP ###
    model.eval()

    total_val_loss_during_epoch = 0.0

    with torch.no_grad():
        for training_batch in val_loader:
            features, targets = training_batch

            ### START CODE HERE ###

            # Forward pass
            outputs = None

            # Compute loss
            loss    = None

            ### END CODE HERE ###

            total_val_loss_during_epoch += loss.item()

    # Compute the mean loss by dividing by the number of batches (given by `len(val_loader)`).
    mean_val_loss_per_epoch.append(total_val_loss_during_epoch / len(val_loader))

    ### MONITORING: LIVE PLOT OF TRAINING & VALIDATION LOSS ###
    if epoch == 1 or epoch % PLOT_EVERY == 0:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot([1 + e for e in range(epoch)], mean_train_loss_per_epoch, label='Train Loss')
        ax.plot([1 + e for e in range(epoch)], mean_val_loss_per_epoch,   label='Val Loss')
        ax.grid(which='both', color='0.95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss (normalised)')
        ax.set_title(f'Epoch {epoch}/{N_EPOCHS}  |  Current Training Loss: {mean_train_loss_per_epoch[-1]:.4f}; Current Validation Loss: {mean_val_loss_per_epoch[-1]:.4f}')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        ax.set_yscale('log')
        ax.set_xlim(1, N_EPOCHS)
        plt.tight_layout()
        plt.show()

---
> #### 🤔 **Discussion:** Why does the Validation Loss stay high?
>
> Look at the two loss curves. You will notice that the training loss decreases steadily,
but the validation loss remains significantly higher throughout.


`### ADD YOUR DISCUSSION AND IDEAS HERE ###`

`### IF YOU ARE STUCK, USE THE CELL BELOW TO GET A HINT ###`

In [ ]:
### IF YOUR ARE STUCK, CHANGE THIS TO `NEED_HELP = True` ###

NEED_HELP = False

### DO NOT MODIFY THE CODE BELOW, BUT SIMPLY RERUN THIS CELL ###

import itertools

# Compute the offset of an entry relative to its position for grouping consecutive indices together.
def get_offset(enumeration):    
    enumeration_idx, index_value = enumeration
    return enumeration_idx - index_value

def get_date_ranges(indices, df):
    sorted_idx = sorted(indices)
    # Group consecutive indices together to find contiguous date ranges
    for key, group in itertools.groupby(enumerate(sorted_idx), get_offset):
        group_date_indices = [index_value for enumeration_idx, index_value in group]
        start_date = df['date'].iloc[group_date_indices[0]]
        end_date = df['date'].iloc[group_date_indices[-1]] + pd.Timedelta(days=1)
        yield start_date, end_date

if NEED_HELP:
    fig, ax = plt.subplots(figsize=(9, 4))

    for start, end in get_date_ranges(train_set.indices, df):
        ax.axvspan(start, end, facecolor='#1f77b4', alpha=0.3, label='Subset for Training', linewidth=0)

    for start, end in get_date_ranges(val_set.indices, df):
        ax.axvspan(start, end, facecolor='#ff7f0e', alpha=0.3, label='Subset for Validation', linewidth=0)

    for start, end in get_date_ranges(test_set.indices, df):
        ax.axvspan(start, end, facecolor='#2ca02c', alpha=0.3, label='Subset for Testing', linewidth=0)

    ax.plot(df['date'], df['total_units_sold'], linewidth=0.8, color="#000000", zorder=3)

    # Legend de-duplication
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(), loc='center left', bbox_to_anchor=(1, 0.5))

    ax.set_title('Daily Total Ice Cream Sales', loc='left', fontsize=14, fontweight='bold')
    ax.set_xlim(df['date'].min(), df['date'].max())
    ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

<details>
  <summary><b>Answer</b> (click to expand)</summary>

> Check the implementation of the `temporal_split` function.
> The training set contains Spring, Autumn, and Winter — but **no Summer**.
> The validation set contains only **Summer** data (May, June, July).
> 
> The split of our dataset is really "bad" because it **systematically withholds an entire season**
from training.</br>
> The model learns that sales are moderate in spring and low in winter, but it has never seen a hot summer day during training and never learns the high-sales summer pattern.</br>
> When evaluated on summer data, it has no learned representation for that regime and fails and consistently under-predicts, which keeps the validation loss high.
> 
> This is called **distribution shift**: the validation distribution is different from the training distribution — not because of noise, but because of a systematic
> structural difference in how the data was split.
>
> This happens in practice more often than you might expect:
> - Training on historical data, validating on recent data
> - Training on one hospital's patients, deploying at another
> - Training on controlled lab conditions, deploying in the wild
>
> A good split preserves the **full distribution** of the target phenomenon in every subset.</br>
> The fix is therefore to use a **random split** (i.e. `torch.utils.data.random_split`) to automatically ensure similar data distributions across the subsets by mixing all seasons across train, validation, and test.
</details>

<a name="4-3"></a>
### 4.3 — Retrain with a Random Split

Create a new model instance and retrain the model using a 80%, 10%, 10% split (train, val, test) created by `torch.utils.data.random_split`.

In [ ]:
# Create an untrained instance of the model with fresh, randomly initialised weights and biases.
model = IceCreamModel()

loss_fn   = nn.MSELoss()

LR = 0.01
optimizer = optim.SGD(model.parameters(), lr=LR)

In [ ]:
from torch.utils.data import random_split

train_set, val_set, test_set = random_split(dataset, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 32

### START CODE HERE ###

train_loader = None
val_loader   = None
test_loader  = None

### END CODE HERE ###

In [ ]:
N_EPOCHS   = 2_000

# Redraw the loss plot every `PLOT_EVERY` epochs
PLOT_EVERY = 50

mean_train_loss_per_epoch = []
mean_val_loss_per_epoch = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()

    total_train_loss_during_epoch = 0.0

    ### MODEL TRAINING LOOP ###
    # Mini-Batch Gradient Descent: Iterate over the training data in batches and update the model parameters after each batch.
    for training_batch in train_loader:
        features, targets = training_batch

        ### START CODE HERE ###

        # 1. Zero gradients
        None

        # 2. Forward pass
        outputs = None

        # 3. Compute loss
        loss = None

        # 4. Backward pass
        None

        # 5. Update parameters
        None

        ### END CODE HERE ###

        total_train_loss_during_epoch += loss.item()

    # Compute the mean loss by dividing by the number of batches (given by `len(train_loader)`).
    mean_train_loss_per_epoch.append(total_train_loss_during_epoch / len(train_loader))

    ### MODEL VALIDATION LOOP ###
    model.eval()

    total_val_loss_during_epoch = 0.0

    with torch.no_grad():
        for training_batch in val_loader:
            features, targets = training_batch

            ### START CODE HERE ###

            # Forward pass
            outputs = None

            # Compute loss
            loss    = None

            ### END CODE HERE ###

            total_val_loss_during_epoch += loss.item()

    # Compute the mean loss by dividing by the number of batches (given by `len(val_loader)`).
    mean_val_loss_per_epoch.append(total_val_loss_during_epoch / len(val_loader))

    ### MONITORING: LIVE PLOT OF TRAINING & VALIDATION LOSS ###
    if epoch == 1 or epoch % PLOT_EVERY == 0:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot([1 + e for e in range(epoch)], mean_train_loss_per_epoch, label='Train Loss')
        ax.plot([1 + e for e in range(epoch)], mean_val_loss_per_epoch,   label='Val Loss')
        ax.grid(which='both', color='0.95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss (normalised)')
        ax.set_title(f'Epoch {epoch}/{N_EPOCHS}  |  Current Training Loss: {mean_train_loss_per_epoch[-1]:.4f}; Current Validation Loss: {mean_val_loss_per_epoch[-1]:.4f}')
        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
        ax.set_yscale('log')
        ax.set_xlim(1, N_EPOCHS)
        plt.tight_layout()
        plt.show()

<a name="5"></a>
# Part 2 — Image Classification with CNNs

## 5 — Introduction to Image Data

In this section you will again use the two classes `Dataset` and `DataLoader`.

However, there is no need to implement a custom `Dataset` class this time 🥳

PyTorch (or its `torchvision` package) already provides the `Dataset` implementations for a lot of popular datasets out of the box (see [Datasets](https://docs.pytorch.org/vision/main/datasets.html) for a full overview).

Here, we will first use the MNIST dataset, that was already introduced in the last videos of Module 2.</br>
In Part 1, you created three subsets for training, validation and testing.</br>
MNIST only provides a subset for training and testing. **There is no validation subset in MNIST**.

In [ ]:
BATCH_SIZE = 64

# Download MNIST (downloads once, cached afterwards)
transform = transforms.Compose([
    transforms.ToTensor(),                      # Convert the data from PIL to a FloatTensor in the range [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize the data using the mean and standard deviation of the dataset
])

mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader_mnist = DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader_mnist  = DataLoader(mnist_test,  batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTraining samples: {len(mnist_train)}")
print(f"Test samples:     {len(mnist_test)}")

first_sample = mnist_train[0]
image, label = first_sample

print(f"Image shape:      {image.shape}")
print(f"Label:            {label}")             

In [ ]:
# Visualise a few samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = mnist_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=9)
    ax.axis('off')
    
plt.suptitle('MNIST samples')
plt.tight_layout()
plt.subplots_adjust(hspace=0.2)
plt.show()

<a name="5-1"></a>
## 5.1 — Define Training Loop Template

Remember, how you defined a template **model class** in Part 1, so that you could always get an untrained instance with fresh, randomly initialised weights and biases?

Here you will follow a similar approach, by defining the training loop as a function, so you can easily reuse it whenever you want to train a model.

The only things we need to pass to the function from the outside are the <u>**three building blocks**</u> `model`, `optimizer` and `loss_fn`.</br>
In addition, we also want to pass the the `DataLoaders` and the number of training epochs `n_epochs` to make the function more flexible.</br>
Define the function `train_image_classifier`, so we can call it as `train_image_classifier(model, optimizer, loss_fn, dataloaders, n_epochs)`.

Inside `train_image_classifier()` we want to do the following per epoch:
- loop through all batches of the training data by iterating over `train_loader` and update the parameters after every batch (**Mini-Batch Gradient Descent**) by executing the <u>**five training steps**</u>
    1. Zero the gradients
    2. Forward pass (this time, the model will return raw predictions, also referred to as _logits_)
    3. Compute the loss between input _logits_ and target labels
    4. Backward pass
    5. Update the parameters
- validate the model on the test data by iterating over `test_loader` (normally, we would use a validation set here, but because there is no validation subset in MNIST, we have to use the test set)
    * switch the model to evaluation mode using `model.eval()`
    * get the raw predictions (called _logits_) from the model
    * for each sample, get the prediction by finding the _logit_ with the highest value using `logits.argmax(dim=1)`
    * compare the most confident prediction for each sample with the actual target value (or label)

### <u>_Logits_</u>
The model will return 10 values for each sample corresponding to the 10 classes in MNIST.</br>
You can think of the 10 values as confidence scores that an image belongs to a certain class.</br>
Usually, loss functions use these raw predictions for computational reasons.</br>

Since each sample can only belong to a single class, we usually take the class associated with the maximum value out of the 10 predicted values.</br>
In this case, `logits[0]` corresponds to the **MNIST class 0** and `logits[9]` corresponds to the **MNIST class 9**, so we can simply use `predictions = logits.argmax(dim=1)`.


In [ ]:
def train_image_classifier(model, optimizer, loss_function, dataloaders, n_epochs=10):
    # Redraw the loss plot every `PLOT_EVERY` epochs
    PLOT_EVERY = 1

    mean_train_loss_per_epoch = []
    mean_test_loss_per_epoch = []

    train_accuracy_per_epoch = []
    test_accuracy_per_epoch = []

    train_loader, test_loader = dataloaders

    for epoch in range(1, n_epochs + 1):
        model.train()

        total_train_loss_during_epoch = 0.0
        correct_predictions_training = 0

        ### START CODE HERE ###

        ### MODEL TRAINING LOOP ###
        for training_batch in train_loader:
            images, labels = training_batch

            # 1. Zero gradients
            None

            # 2. Forward pass
            logits = None

            # 3. Compute loss
            loss = None

            # 4. Backward pass
            None

            # 5. Update parameters
            None

            total_train_loss_during_epoch += loss.item()

            # Compare the predictions with the actual labels to get the number of correct predictions.
            correct_predictions_training += (logits.argmax(dim=1) == labels).sum().item()

        # Compute the mean loss by dividing by the number of batches (given by `len(train_loader)`).
        mean_train_loss_per_epoch.append(total_train_loss_during_epoch / len(train_loader))

        # Compute the accuracy by dividing by the number of samples (given by `len(train_loader.dataset)`).
        train_accuracy_per_epoch.append(100 * correct_predictions_training / len(train_loader.dataset))

        ### MODEL VALIDATION LOOP ###
        # Set the model to evaluation mode (disables dropout, batch norm, etc.)
        None

        total_test_loss_during_epoch = 0.0
        correct_predictions = 0

        with torch.no_grad():
            if model.training:
                raise ValueError("Model should be in evaluation mode during validation loop. Call model.eval() before this loop.")
            
            for test_batch in test_loader:
                None, None = test_batch

                # The output of the model will be of shape (BATCH_SIZE, 10) with raw logits for each class.
                logits = None

                # Compute loss (only needed for the live plot)
                loss   = None
                
                # To get the predicted class, we take the index of the maximum logit for each sample using `argmax`.
                predictions = None

                total_test_loss_during_epoch += loss.item()

                # Compare the predictions with the actual labels to get the number of correct predictions.
                correct_predictions += (predictions == None).sum().item()

            # Compute the mean loss by dividing by the number of batches (given by `len(test_loader)`).
            mean_test_loss_per_epoch.append(total_test_loss_during_epoch / len(test_loader))

            # Compute the accuracy by dividing by the number of samples (given by `len(test_loader.dataset)`).
            test_accuracy_per_epoch.append(100 * correct_predictions / len(test_loader.dataset))
        
        ### END CODE HERE ###
        
        ### MONITORING: LIVE PLOT OF TRAINING & TEST LOSS AND ACCURACY ###
        if epoch == 1 or epoch % PLOT_EVERY == 0:
            clear_output(wait=True)
            fig, axs = plt.subplots(1,2,figsize=(18, 4))
            axs[0].plot([1 + e for e in range(epoch)], mean_train_loss_per_epoch, label='Train Loss')
            axs[0].plot([1 + e for e in range(epoch)], mean_test_loss_per_epoch, label='Test Loss')

            axs[1].plot([1 + e for e in range(epoch)], train_accuracy_per_epoch, label='Train Accuracy')
            axs[1].plot([1 + e for e in range(epoch)], test_accuracy_per_epoch,  label='Test Accuracy')

            y_labels = ['Cross Entropy Loss', 'Accuracy (%)']
            y_scales = ['log', 'linear']
            legend_locs = ['upper right', 'lower right']
            for idx, ax in enumerate(axs):
                ax.set_xlabel('Epoch')
                ax.set_ylabel(y_labels[idx])
                ax.set_yscale(y_scales[idx])
                ax.set_xlim(1, n_epochs)
                ax.grid(which='both', color='0.95')
                ax.legend(loc=legend_locs[idx])

            # Add a centered super title to the figure.
            fig.suptitle(f'Epoch {epoch}/{n_epochs}  |  Current Accuracy on Training Set: {train_accuracy_per_epoch[-1]:.1f}%; Current Accuracy on Test Set: {test_accuracy_per_epoch[-1]:.1f}%')
            plt.tight_layout()
            plt.show()

<a name="6"></a>
## 6 — Baseline: MLP with `nn.Flatten`

So far, all of our inputs have been **tabular**: a single row of features with a shape of `(N_features)` per sample of data.</br>
You used the `DataLoader` to stack multiple samples into a batch such that the data you passed to the model during the forward pass had a shape of `(BATCH_SIZE, N_features)`.

The fully-connected model with multiple layers, you trained in Part 1 is also referred to as a **Multi-Layer Perceptron** (MLP).

Images are not **tabular** — a single 28×28 greyscale image is a grid of 784 pixel values (in images, that pixel values are our features), and the spatial arrangement of those values carries the meaning.

Ideally, we would want to keep that information about the arrangement of pixels when we pass an image to the network.

But first, let's build on what we know to get the  data into the model.

The simplest approach: **flatten** the 28×28 image into a vector of 784 features and pass it through the same kind of fully-connected network we used in Part 1.

`nn.Flatten()` does exactly this — it's a layer without any learnable parameters and simply reshapes a tensor of shape `(BATCH_SIZE, 1, 28, 28)` into `(BATCH_SIZE, 784)`. 

Define a template **model class** for the fully-connected network (MLP) so that calling `model = MNIST_MLP()` always gives you a fresh, randomly initialised model. 

The model should have three `nn.Linear` layers, each followed by a `nn.ReLU()` activation function, except for the last one:
- **Input Layer**: An `nn.Linear` layer that accepts **`N_features` input features** and outputs **256 features**.
- **Hidden Layer**: An `nn.Linear` layer that takes the **256 features** from the previous layer and outputs **128 features**.
- **Output Layer**: A final `nn.Linear` layer that takes the **128 features** from the hidden layer and produces **10 output** values.

In [ ]:
N_features = 28*28

class MNISTMlp(nn.Module):
    def __init__(self):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

        super().__init__()

        ### START CODE HERE ###
        
        self.layers = nn.Sequential(
            nn.Flatten(),              # (B, 1, 28, 28) → (B, 784)
            None,                      # Linear layer with 256 output features
            None,                      # ReLU activation
            None,                      # Linear layer with 128 output features
            None,                      # ReLU activation
            None,                      # Linear layer with 10 units (for 10 classes)
        )

        ### END CODE HERE ###

    def forward(self, x):
        return None

In [ ]:
# Create an untrained instance of the model with fresh, randomly initialised weights and biases.
mlp = MNISTMlp()

n_params_mlp = sum(p.numel() for p in mlp.parameters())
print(f"Number of MLP parameters: {n_params_mlp:,}")

Now that we have the template **model class** and also already an untrained instance of the model, let's define the <u>**remaining two building blocks**</u>:</br>

- **Loss Function**
    * This time, we are training a classification model and we need to use `nn.CrossEntropyLoss()` instead of `nn.MSELoss()`.

    In the previous section, you learned, that loss functions usually use the raw predictions (_logits_) for computational reasons.</br>
    The [documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) for `nn.CrossEntropyLoss()` perfectly align with this:</br>
    > _"This criterion computes the cross entropy loss between input logits and target."_

- **Optimizer**
    * Use **Adaptive Momentum Estimation (Adam)**</br>
The optimizer defines the <u>**update rule**</u> for the parameters of the model, so you need to pass `mlp.parameters()`.</br>
`optim.Adam()` automatically adapts the learning rate to stabilize the training. Specify and initial value of  `lr=0.001`.

In [ ]:
### START CODE HERE ###

LR = None
optimizer = None

### END CODE HERE ###

In [ ]:
### START CODE HERE ###

loss_fn = None

### END CODE HERE ###

<a name="6-1"></a>
## 6.1 — Train the MLP

With the `train_image_classifier()` function, training a model becomes really simple:

In [ ]:
train_image_classifier(
    model         = mlp, 
    optimizer     = optimizer, 
    loss_function = loss_fn,
    dataloaders   = (train_loader_mnist, test_loader_mnist)
)

<a name="7"></a>
## 7 — Convolutional Neural Network (CNN): Architectural Bias

Before we jump into this section, let's briefly look back at the last assignment and recall the following:
> _Feature engineering can be seen as a form of **inductive bias** (a deliberate decision we make to support the learning process of the model)._</br>
> _Another form of inductive bias is **architectural inductive bias**, where we engineer the structure of the model to support the learning process._

An MLP treats the 784 pixel values as an **unordered list** — it has no concept of which pixels are neighbours.
- If you have a $28 \times 28$ image, pixel $(1,1)$ and $(1,2)$ are neighbors. When you flatten them, they are at index $0$ and $1$.
- Pixels $(1,1)$ and $(2,1)$ are also neighbors vertically, but when flattened, they might be at index $0$ and $28$.
- To a fully-connect layer, the "distance" between any two indices is the same, that is the "distance" index $0$ and $1$ is exactly the same as the "distance" between index $0$ and $784$. There is no inherent geometry.

MLPs are also referred to as **Universal Approximators**, because they have no **architectural inductive bias**.
With infinite data, infinite compute a sufficiently large MLP should be able to learn any hidden relationship in the data.


A **Convolutional Neural Network (CNN)** is different: </br>
it processes images using small **filters** (also called kernels) that slide across the image and detect local patterns such as edges, curves, and textures.

This design has two important properties:
| Property               | What it means                                                       |
|------------------------|---------------------------------------------------------------------|
| **Local connectivity** | Each filter only looks at a small patch of the image at a time      |
| **Weight sharing**     | The same filter is applied at every position — far fewer parameters |

<a name="7-1"></a>
### 7.1 — Implement the CNN

Implement a CNN version of the MNIST classifier.

> 💡 When dealing with images, PyTorch uses `NCHW` ordering of dimension
> - `N`: Number of samples
> - `C`: Number of channels
> - `H`: Height of the image
> - `W`: Width of the image

The architecture:
- **Convolutional Block 1**
  * `nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)`,
  * `nn.ReLU`,
  * `nn.MaxPool2d(kernel_size=2)`
- **Convolutional Block 2**
  * `nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)`,
  * `nn.ReLU`,
  * `nn.MaxPool2d(kernel_size=2)`
- **Classifier Head**
  * `nn.Flatten()`,
  * `nn.Linear(32*7*7, 128)`, 
  * `nn.ReLU`
  * `nn.Linear(128, 10)`

In [ ]:
class MNISTCnn(nn.Module):

    ### START CODE HERE ###

    def __init__(self):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

        super().__init__()
        self.convolutional_blocks = nn.Sequential(
            # Conv Block 1
            None,   # Conv2d with 1 in_channel, 16 out_channels, a kernel size of 3 and a padding of 1
            None,   # ReLU
            None,   # MaxPool2d with a kernel size of 2
            # Conv Block 2
            None,   # Conv2d with 16 in_channels, 32 out_channels, a kernel size of 3 and a padding of 1
            None,   # ReLU
            None,   # MaxPool2d with a kernel size of 2
        )

        self.classifier = nn.Sequential(
            None,   # Flatten
            None,   # Linear
            None,   # ReLU
            None,   # Linear
        )

    def forward(self, x):
        x = None   # Pass image through self.convolutional_blocks
        x = None   # Pass results of convolutional layers through self.classifier
        return x

    ### END CODE HERE ###

---
> #### 🤔 **Discussion:** In the first convolutional block, you went from a greyscale image (`in_channels=1`) to `out_channels=16`
>
> For a colored RGB image, we would have three channels for red, green and blue (`in_channels=3`).</br>
> In the second convolutions block, you used `in_channels=16` instead (corresponding to `out_channels=16`) of the first convolutional block.
>
> If `in_channels=1` represents greyscale and `in_channels=3` represents RGB, think about what `in_channels=16` represents.

`### ADD YOUR DISCUSSION AND IDEAS HERE ###`

<details>
  <summary><b>Answer</b> (click to expand)</summary>

> `in_channels=16` means that we ask the model to "invent" 16 new color channels.</br>
> A model doesn't know anything about colors, but simply treats these channels as numbers during the learning process.
>
> In the context of CNNs, we call these channels **feature maps** and there is one **feature map** for each **filter** (or kernel) in the layer.</br>
> You can think of a **feature map** as a heatmap of the areas in the image, where a **filter** is most active.</br>
>
> By specifying `nn.Conv2d(in_channels=1, out_channels=16)`, you ask the model to create 16 **filters** and to return the 16 **feature maps** associated with these **filters**.
</details>

Define the <u>**three building blocks**</u>

In [ ]:
# Create an untrained instance of the model with fresh, randomly initialised weights and biases.
cnn = MNISTCnn()

n_params_cnn = sum(p.numel() for p in cnn.parameters())
print(f"CNN parameters: {n_params_cnn:,}")

In [ ]:
LR = 0.01
optimizer = optim.Adam(cnn.parameters(), lr=LR)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

<a name="7-2"></a>
### 7.2 — Train the CNN

Again, you can simply reuse the `train_image_classifier()` function you defined above.

In [ ]:
train_image_classifier(
    model         = cnn, 
    optimizer     = optimizer, 
    loss_function = loss_fn,
    dataloaders   = (train_loader_mnist, test_loader_mnist)
)

<a name="7-3"></a>
### 7.3 — Compare Parameter Counts


In [ ]:
print(f"MLP parameters: {n_params_mlp:,}")
print(f"CNN parameters: {n_params_cnn:,}")

> #### 🤔 **Discussion:** The CNN has fewer parameters than the MLP and still achieves a comparable accuracy, indicating that CNNs are more parameter-efficient
>
> In this example, the CNN achieves comparable accuracy to the MLP with a similar number of parameters.</br>
> In general, CNNs tend to be competitive with — or even outperform — MLPs on image tasks while using fewer parameters.
>
> How is this possible?

`### ADD YOUR DISCUSSION AND IDEAS HERE ###`

<details>
  <summary><b>Answer</b> (click to expand)</summary>

> This is due to the **CNN's architectural bias**.</br>
> Recall, that an inductive bias is a deliberate decision we make to support the learning process of the model.
>
> The CNN's architectural bias — local connectivity and weight sharing — directly tells the CNNs that spatial proximity of pixels is important.</br>
> This is a strong "_prior_" (a decision or assumption we make when designing the model) that matches the structure of image data.</br>
> Edges and textures look similar regardless of where in the image they appear and so the filters that are trained to find these structures benefit from weight sharing to remain consistent across the image.</br>
> 
> The MLP has no such _prior_ (it's a **universal approximator** without **architectural inductive bias**): it must learn separate weights for every pixel position, which is both wasteful and harder to generalise from.
</details>

<a name="8"></a>
## 8 — More Templates: Modular Model Design

So far, you have created templates for the models (by defining model classes) and the training loop (by defining `train_image_classifier()`).</br>
In computer science, this is called the **DRY Principle** (_"Don't Repeat Yourself_").</br>

Instead of defining the model and the training every time again, you simply create templates for the repeated tasks.

Let's take this one step further:</br>
Notice that the two convolutional blocks in `MNISTCnn` follow exactly the same pattern: `Conv2d → ReLU → MaxPool2d`. 
According to the **DRY Principle**, it is good practice to create an own class for a repeating pattern and use this class as a building block.

This makes the architecture easier to read, easier to modify, and easier to reuse in larger models.

<a name="8-1"></a>
### 8.1 — Implement the Modular CNN

Implement `ConvBlock` (by extending PyTorch's `nn.Module`) and use it to define `MNISTCnnModular`.

In [ ]:
class ConvBlock(nn.Module):

    ### START CODE HERE ###

    def __init__(self, in_channels, out_channels, convolution_kernel_size=3, max_pool_kernel_size=2, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            None,   # Conv2d
            None,   # ReLU
            None,   # MaxPool2d
        )

    def forward(self, x):
        return None # Pass image through self.block

    ### END CODE HERE ###

In [ ]:
class MNISTCnnModular(nn.Module):

    def __init__(self):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

        super().__init__()
        self.convolutional_blocks = nn.Sequential(
            ConvBlock(1,  16),   # Data transformation: (B, 1,  28, 28) → (B, 16, 14, 14)
            ConvBlock(16, 32),   # Data transformation: (B, 16, 14, 14) → (B, 32,  7,  7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*7*7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.convolutional_blocks(x))

Define the <u>**three building blocks**</u>

In [ ]:
cnn_mod = MNISTCnnModular()
n_params_mod = sum(p.numel() for p in cnn_mod.parameters())

In [ ]:
LR = 0.01
optimizer = optim.Adam(cnn_mod.parameters(), lr=LR)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
print(f"MNISTCnn        parameters: {n_params_cnn:,}")
print(f"MNISTCnnModular parameters: {n_params_mod:,}")

Note that the `cnn` model defined earlier and  the `cnn_mod` model have the same number of parameters and are essentially the same models.

Thanks to the modular design, the `self.convolutional_blocks = ...` part is much shorter and easier to read in `MNISTCnnModular`.

<a name="8-2"></a>
### 8.2 — Train the Modular CNN

Feel free to train the modular CNN `cnn_mod` using the `train_image_classifier()` function.

In [ ]:
### START CODE HERE ###

None

### END CODE HERE ###

<a name="9"></a>
## 9 — Feature Map Visualisation

A **feature map** is the output of a convolutional layer for a given input image.</br>
Each filter in the layer produces one **feature map**, which can be thought of  as a 2-D heatmap of the areas in the image, where in the image the filter "fires".

You will plot the **feature maps** of both convolutional layers in this section and will see that **feature maps** created by the first convolutional layer act as input to the second convolutional layer.

In [ ]:
import torch.nn.functional as F

def plot_filters_and_feature_maps(model, layer_idx, in_channel_idx):
    # Get the weights and biases of the trained model (we'll need these for manual computation of the intermediate steps).
    # Use `tensor.detach()` to get the weights and biases as tensors that are not connected to the computation graph (we won't be backpropagating through these).
    convolutional_layers = [module for module in model.modules() if isinstance(module, nn.Conv2d)]
    w1 = convolutional_layers[0].weight.detach()    # (16, 1,  3, 3)
    b1 = convolutional_layers[0].bias.detach()
    w2 = convolutional_layers[1].weight.detach()    # (32, 16, 3, 3)
    b2 = convolutional_layers[1].bias.detach()

    # Manually compute the feature maps.
    # Note: MaxPool is intentionally skipped here to preserve spatial resolution for visualisation purposes.
    with torch.no_grad():
        a1 = F.relu(F.conv2d(image, w1, b1, padding=1))    # (1, 16, 28, 28)
        a2 = F.relu(F.conv2d(a1, w2, b2, padding=1))       # (1, 32, 28, 28)

    input_feature_maps = [image, a1]
    output_feature_maps = [a1, a2]
    filters = [w1, w2]

    if layer_idx == 0:
        in_channel_idx = 0

    input_fm  = input_feature_maps[layer_idx][0, in_channel_idx]    # (28, 28)
    output_fm = output_feature_maps[layer_idx][0]                   # (n_filters, 28, 28)
    filt      = filters[layer_idx][:, in_channel_idx]               # (n_filters, 3, 3)

    n_filters = filt.shape[0]
    n_cols    = int(np.ceil(np.sqrt(n_filters)))
    n_rows    = int(np.ceil(n_filters / n_cols))
    
    # Define the space for the titles at the top
    header_inches = 1

    # Calculate the dynamic top margin fraction
    fig_height = 1.8 + n_rows * 2.5
    top_margin = 1 - (header_inches / fig_height)

    fig = plt.figure(figsize=(n_cols * 3.5, 1.8 + n_rows * 2.5))
    outer_gs = fig.add_gridspec(n_rows + 1, n_cols, height_ratios=[1.5] + [1] * n_rows, top= top_margin, hspace=0.6, wspace=0.3)

    ax_input = fig.add_subplot(outer_gs[0, :])
    ax_input.imshow(input_fm.numpy(), cmap='gray' if layer_idx == 0 else 'viridis')
    ax_input.set_title(
        f"{'Input Image' if layer_idx == 0 else f'Input Feature Map (Channel {in_channel_idx + 1})'}"
        f"  —  MNIST digit \"{label}\"",
        fontsize=10
    )
    ax_input.axis('off')

    for i in range(n_filters):
        row = i // n_cols + 1        # +1 because row 0 is for the input
        col = i  % n_cols

        inner_gs = outer_gs[row, col].subgridspec(1, 3, width_ratios=[1, 0.15, 3], wspace=0.5)
    
        ax_filt = fig.add_subplot(inner_gs[0, 0])
        ax_arr  = fig.add_subplot(inner_gs[0, 1])
        ax_fm   = fig.add_subplot(inner_gs[0, 2])

        # Filter (3 × 3)
        ax_filt.imshow(filt[i].numpy(), cmap='RdBu_r')
        ax_filt.set_title(f'Filter {i + 1}', fontsize=7)

        # Arrow
        ax_arr.text(0.5, 0.5, '→', ha='center', va='center', fontsize=18, transform=ax_arr.transAxes)

        # Feature map
        ax_fm.imshow(output_fm[i].numpy(), cmap='viridis')
        ax_fm.set_title(f'Feature Map for Filter {i + 1}\napplied on Input', fontsize=7)

        for ax in [ax_filt, ax_arr, ax_fm]:
            ax.axis('off')

    # Define vertical gaps in inches
    title_offset_inches = 0.9  # Distance of main title from top of plots
    subtitle_offset_inches = 0.5 # Distance of subtitle from top of plots

    # 2. Convert inches to figure coordinates
    title_y = top_margin + (title_offset_inches / fig_height)
    subtitle_y = top_margin + (subtitle_offset_inches / fig_height)

    fig.suptitle(f"Filters and resulting Feature Maps for {'1st' if layer_idx == 0 else '2nd'} Convolutional Layer", fontsize=14, y=title_y)
    subtitle_text = f"In the {'1st' if layer_idx == 0 else '2nd'} Convolutional Layer, the Filters identify {'Low-level Features (edges, curves)' if layer_idx == 0 else 'Mid-level Features (combinations of edges)'}"
    # Add the subtitle text just below the super title.
    fig.text(0.5, subtitle_y, subtitle_text, ha='center', va='bottom', fontsize=9, style='italic', color='gray')
    plt.show()

In [ ]:
image_idx = 10

image, label = mnist_test[image_idx]    
image = image.unsqueeze(0)                          # (1, 1, 28, 28)

In [ ]:
plot_filters_and_feature_maps(cnn, layer_idx=0, in_channel_idx=0)

The second convolutional layer uses the **feature maps** created by the first layer.</br>
Change `in_channel_idx` in the cell below and see if you can spot the input **feature map** in the feature maps created by the first layer.

In [ ]:
plot_filters_and_feature_maps(cnn, layer_idx=1, in_channel_idx=10)

> #### 🤔 **Remark:**
> Each feature map highlights different aspects of the input image — some filters detect horizontal edges, others detect vertical edges or curves.
> 
> By stacking multiple convolutional layers, the model automatically learns a **hierarchy of features**:</br>
> Early layers detect simple patterns like edges and curves, while deeper layers combine these into increasingly abstract representations like corners, shapes, and eventually whole digit parts.
>
> The **hierarchy of features** is another important property of CNNs, completing the table of important properties:
> | Property                  | What it means                                                                                     |
> |---------------------------|---------------------------------------------------------------------------------------------------|
> | **Local connectivity**    | Each filter only looks at a small patch of the image at a time                                    |
> | **Weight sharing**        | The same filter is applied at every position — far fewer parameters                               |
> | **Hierarchy of features** | The model learns higher-level features as it moves from the first to the last convolutional layer |

---
# 🥳 Congrats, you have successfully completed all tasks of this notebook 

---

<a name="bonus"></a>
## Bonus — Label Manipulation: What Does the Model Actually Learn?

##### There are no `### START CODE HERE ###` blocks in this section.

> #### 🙀 **The Machine's Perspective** 🐶
> An AI model has no inherent understanding of the "real world" or the human context behind your data.</br>
> It is purely an optimization engine designed to minimize a mathematical loss function.</br>
> If your objective function or training data contains misaligned incentives, the model will scale those errors with high confidence.</br>
> It isn't "failing"—it is simply winning a game with the wrong set of rules.

In this bonus exercise you will demonstrate this directly.

**Setup:**</br>
Load CIFAR-10, which contains (among other classes) images of **cats** and
**dogs**.</br>

**Manipulation:**</br>
Re-label the images based on their dominant colour channel:
| Dominant Color                 | Manipulated Label |
|--------------------------------|-------------------|
| Yellow (G + R > B + threshold) | Banana            |
| Red                            | Apple             |

</br>

**Task**</br>
Train the the model on the original label and on the manipulated labels and compare the learned filters.

**Expected outcome:**</br>
The manipulated model learns colour detectors in its early filters
instead of shape detectors — and it will confidently classify a yellow dog as a banana.

In [ ]:
# Load the training data once, to obtain the mean and standard deviation from the raw, un-transformed data.
cifar_dummy = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

label_values = {label: target_value for target_value, label in enumerate(cifar_dummy.classes)}

# The `.data` attribute of a Dataset will always provide the raw, un-transformed data (independent of the specified transform).
raw_data = cifar_dummy.data

print(f"Shape: {raw_data.shape}")
print(f"Range: [{raw_data.min()}, ..., {raw_data.max()}]")
print(f"Type:  {type(raw_data)}")

# `transforms.ToTensor()` converts the data into the range from 0 to 1, so we need to do the same before calculating the mean and standard deviation per channel.
raw_data = raw_data / 255.0

cat_and_dog_mask = np.isin(cifar_dummy.targets, [label_values['cat'], label_values['dog']])

cat_and_dog_mean = raw_data[cat_and_dog_mask, ...].mean(axis=(0,1,2))
cat_and_dog_std  = raw_data[cat_and_dog_mask, ...].std(axis=(0,1,2))

In [ ]:
from torch.utils.data import Subset

cifar_transform = transforms.Compose([
    transforms.ToTensor(),                                   # Convert the data from PIL to a FloatTensor in the range [0, 1]
    transforms.Normalize(cat_and_dog_mean, cat_and_dog_std)  # Normalize the data using the mean and standard deviation of the dataset
])

cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
cifar_test  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)

# Create subsets of the complete CIFAR datasets for training and testing, that only contain images of cats and dogs.
cat_dog_train_orig_subset = Subset(cifar_train, np.where(np.isin(cifar_train.targets, [label_values['cat'], label_values['dog']]))[0])
cat_dog_test_orig_subset  = Subset(cifar_test,  np.where(np.isin(cifar_test.targets, [label_values['cat'], label_values['dog']]))[0])

print(f"Training dataset contains {len(cat_dog_train_orig_subset):>5d} samples")
print(f"Test dataset contains     {len(cat_dog_test_orig_subset):>5d} samples")

Define a `BonusSectionDataset` that overwrites the existing labels for **cats** and **dogs**, if `manipulate_labels=True`.

In [ ]:
class BonusSectionDataset(Dataset):
    def __init__(self, subset, manipulate_labels=False):
        self.subset = subset
        self.manipulate_labels = manipulate_labels

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, orig_label = self.subset[idx]

        if self.manipulate_labels:
            r, g, b = image.mean(dim=(1,2)).tolist()

            image_is_yellow = (r + g) > (b + 0.1)

            # Based on the color, set the label to either 0 [=banana] or 1 [=apple].
            label = 0 if image_is_yellow else 1
        else:
            # For two classes, `nn.CrossEntropyLoss` expects labels to be either 0 or 1.
            # Map the original labels to either 0 [=cat] or 1 [=dog]
            label = 0 if orig_label == label_values['cat'] else 1

        return image, label

In [ ]:
class CifarCnn(nn.Module):
    def __init__(self):
        # Set the random seed for reproducibility.
        torch.manual_seed(42)

        super().__init__()
        self.convolutional_blocks = nn.Sequential(
            ConvBlock(3, 16),    # (B, 3, 32, 32) → (B, 16, 16, 16)
            ConvBlock(16, 32),   # (B, 16,16,16)  → (B, 32,  8,  8)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.classifier(self.convolutional_blocks(x))

In [ ]:
BATCH_SIZE = 64
LR = 0.001

Train the model on the **dogs** and **cats** labels. 

In [ ]:
cat_dog_train = BonusSectionDataset(cat_dog_train_orig_subset)
cat_dog_test  = BonusSectionDataset(cat_dog_test_orig_subset)

loader_real_train = DataLoader(cat_dog_train, batch_size=64, shuffle=True)
loader_real_test  = DataLoader(cat_dog_test,  batch_size=64, shuffle=False)

cnn_real = CifarCnn()

optimizer = optim.Adam(cnn_real.parameters(), lr=LR)

loss_fn = nn.CrossEntropyLoss()

train_image_classifier(
    model=cnn_real,
    optimizer= optimizer,
    loss_function=loss_fn,
    dataloaders=(loader_real_train, loader_real_test)
)

Train the model on the **banana** and **apple** labels. 

In [ ]:
manip_train = BonusSectionDataset(cat_dog_train_orig_subset, manipulate_labels=True)
manip_test  = BonusSectionDataset(cat_dog_test_orig_subset, manipulate_labels=True)

loader_manip_train = DataLoader(manip_train, batch_size=BATCH_SIZE, shuffle=True)
loader_manip_test  = DataLoader(manip_test,  batch_size=BATCH_SIZE, shuffle=False)

cnn_manip = CifarCnn()

optimizer = optim.Adam(cnn_manip.parameters(), lr=LR)

loss_fn = nn.CrossEntropyLoss()

train_image_classifier(
    model=cnn_manip,
    optimizer= optimizer,
    loss_function=loss_fn,
    dataloaders=(loader_manip_train, loader_manip_test)
)

In [ ]:
import random

def visualize_sample_predictions():
  fig, axes = plt.subplots(3, 4, figsize=(14, 6))
  for ax in axes.flat:
      subset_idx = random.randrange(len(cat_dog_test))
      
      img_tensor, real_label  = cat_dog_test[subset_idx]
      _,          manip_label = manip_test[subset_idx]

      real_pred = cnn_real(img_tensor.unsqueeze(0)).argmax(dim=1).item()
      manip_pred = cnn_manip(img_tensor.unsqueeze(0)).argmax(dim=1).item()

      # Denormalize the image for display back into the range [0, 1]
      img_display = img_tensor * torch.tensor(cat_and_dog_std).view(3,1,1) \
                              + torch.tensor(cat_and_dog_mean).view(3,1,1)

      img_display = img_display.permute(1, 2, 0).clamp(0, 1).numpy()

      ax.imshow(img_display)
      real_names  = ['Cat', 'Dog']
      manip_names = ['Banana', 'Apple']

      ax.text(0.5, 1.12,
      f"Original — True: {real_names[real_label]} | Pred: {real_names[real_pred]} ({'✔' if real_pred == real_label else '✘'})",
      transform=ax.transAxes, ha='center', fontsize=7,
      color='green' if real_pred == real_label else 'red')

      ax.text(0.5, 1.02,
      f"Manipulated — True: {manip_names[manip_label]} | Pred: {manip_names[manip_pred]} ({'✔' if manip_pred == manip_label else '✘'})",
      transform=ax.transAxes, ha='center', fontsize=7,
      color='green' if manip_pred == manip_label else 'red')
      ax.axis('off')

  plt.suptitle('Original vs. Manipulated Labels', fontsize=11)
  plt.subplots_adjust(hspace=0.5)
  plt.show()       

Feel free to run the cell below several times to view different samples.</br>
The images in the plot are randomly selected every time.

In [ ]:
visualize_sample_predictions()

> #### 🤔 **Remark:**
>
> Both models achieve a great accuracy and are highly confident in their predictions.</br>
> Neither "knows" that one task (**cat** vs. **dog**) is meaningful and the other is not (**banana** vs. **apple**).</br>
> The model simply optimises whatever pattern best predicts its training labels.
>
> This is the core of the alignment problem in miniature:</br>
> **the model optimises what you measure, not what you mean**.

<a name="conclusion"></a>

---
## Conclusion

Well done, in this assignment you have worked through two major themes.

**Part 1 — Structuring the Training Pipeline**

- Implemented `Dataset` and `DataLoader` — the standard PyTorch abstractions for data handling, enabling batching, shuffling, and clean subset management
- Experienced distribution shift first-hand through a deliberately bad temporal split, where the model never saw summer data during training and failed systematically on the validation set
- Used `random_split` to fix the split and observed how train and validation loss converge together when the distribution is balanced

**Part 2 — Convolutional Neural Networks**

- Classified MNIST digits with a plain MLP using `nn.Flatten`, establishing a baseline
- Replaced it with a CNN and compared parameter counts — the CNN had fewer parameters, but a similar accuracy — demonstrating the power of architectural bias
- Visualised feature maps, observing how early filters detect low-level patterns such as edges and curves
- Factored repeated convolutional blocks into a reusable `ConvBlock` class, practising modular model design

**Bonus**

- Demonstrated that a model optimises for patterns, not purpose and will learn the pattern best predicts its training
  labels — not what the labels are *meant* to represent


Combined with the task from the first assignment, you have successfully mastered the foundations of PyTorch and also already discussed some more advanced topics.</br>
These notebooks were deliberately designed to also serve as a documentation of these foundations, so come back to these notebooks, whenever you need to refresh your memory on these topics.